# Compare Benchmark Charts

**Purpose:**
- Read Notebook 2 artifacts
- Generate comparison tables and charts without loading models
- Enforce schema validation

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
if _cwd.name != "notebooks":
    sys.path.insert(0, str(_cwd / "notebooks"))

from utils import setup_root
ROOT = setup_root()

## Locate latest completed run

In [ ]:
import json

try:
    with open(ROOT / "results/charts/notebook_demo/latest_run.json", "r") as f:
        latest_run = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError("latest_run.json not found. Please run Notebook 02 first.")

RUN_ID = latest_run["run_id"]
DATASET = latest_run["dataset"]
print(f"Latest run ID: {RUN_ID} (Dataset: {DATASET})")

## Load and validate artifacts

In [ ]:
import pandas as pd

CSV_PATH = ROOT / latest_run["table_dir"] / "comparison.csv"
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Missing comparison.csv at {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

required_cols = ["condition", "display_name", "input_tokens", "output_tokens", "t_e2e_ms", "generation_tok_s", "peak_vram_gib"]
for c in required_cols:
    assert c in df.columns, f"Missing required column: {c}"
print("Artifact loaded and validated successfully.")

## Show concise comparison table

In [ ]:
metrics_mapping = [
    ("input_tokens", "Input tokens", "{:.0f}"),
    ("compressed_tokens", "Compressed tokens", "{:.0f}"),
    ("compression_ratio", "Compression ratio", "{:.2f}"),
    ("t_compress_ms", "T_compress", "{:.2f}"),
    ("t_prefill_ms", "T_prefill", "{:.2f}"),
    ("t_generation_ms", "T_generation", "{:.2f}"),
    ("t_e2e_ms", "T_e2e", "{:.2f}"),
    ("generation_tok_s", "Generation tok/s", "{:.2f}"),
    ("e2e_tok_s", "E2E tok/s", "{:.2f}"),
    ("peak_vram_gib", "Peak VRAM", "{:.2f}"),
    ("run_status", "Status", "{}")
]

formatted_data = {}
conditions = ["baseline_ar", "dflash_r1", "cc_dflash_r2"]
cond_names = {
    "baseline_ar": "Baseline-AR",
    "dflash_r1": "DFlash-R1",
    "cc_dflash_r2": "CC-DFlash-R2"
}

for c in conditions:
    formatted_data[cond_names[c]] = []

metric_names = []
for col_name, display_label, fmt in metrics_mapping:
    metric_names.append(display_label)
    for c in conditions:
        row_match = df[df["condition"] == c]
        if row_match.empty:
            val = "N/A"
        else:
            val = row_match.iloc[0].get(col_name)
            if pd.isna(val) or val == "" or val == "—":
                val = "—"
        
        if c in ["baseline_ar", "dflash_r1"] and col_name in ["compressed_tokens", "compression_ratio"]:
            val = "—"
        elif c in ["baseline_ar", "dflash_r1"] and col_name == "t_compress_ms":
            val = "0"
        
        if val != "—" and val != "N/A":
            try:
                val = fmt.format(float(val))
            except ValueError:
                val = fmt.format(val)
        
        formatted_data[cond_names[c]].append(val)

df_pivot = pd.DataFrame(formatted_data, index=metric_names)
display(df_pivot)

## Build composite dashboard

In [ ]:
import matplotlib.pyplot as plt
from ccdf.demo.charting import build_three_version_dashboard

FIGURE_DIR = ROOT / latest_run["figure_dir"]
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

dashboard_figure = build_three_version_dashboard(
    df,
    dataset=DATASET,
    run_id=RUN_ID
)

## Display and save composite dashboard

In [ ]:
dashboard_path = FIGURE_DIR / "three_version_comparison_dashboard.png"
dashboard_figure.savefig(
    dashboard_path,
    dpi=180,
    bbox_inches="tight"
)

display(dashboard_figure)
print(f"Saved dashboard: {dashboard_path}")

plt.show()
plt.close(dashboard_figure)

## Final Interpretation and Output Path

- The composite dashboard has been saved to: `results/charts/notebook_demo/{{RUN_ID}}/charts/three_version_comparison_dashboard.png`.
- Quality indicators on QMSum do not claim semantic correctness. The semantic correctness of the target model's output remains a final limitation.